<a href="https://colab.research.google.com/github/akkajoe/Projects/blob/main/Predicting_a_Pulsar_Star.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting a Pulsar Star
Pulsar stars are a rare type of Neutron stars that produce radio emissions detectable on Earth. They are of considerable scientific interest as probes of space-time, the interstellar medium, and states of matter.
We are dealing with a binary class problem. Here, the legitimate pulsar examples are a minority positive class (less in numbers), and the remaining examples are a majority negative class. The class labels used are 0 (negative class) and 1 (positive class).



# Objective
To Deploy The Xgboost Classifier Prediction Model To Detect Which Stars Are Pulsar Stars Amongst The Neutron Stars

# Attributes
8 Attributes to determine label

1. The mean of the integrated profile.

2. The standard deviation of the integrated profile.

3. Excess kurtosis of the integrated profile.

4. The skewness of the integrated profile.

5. The mean of the DM-SNR curve.

6. The standard deviation of the DM-SNR curve.

7. Excess kurtosis of the DM-SNR curve.

8. The skewness of the DM-SNR curve.

Source: https://archive.ics.uci.edu/ml/datasets/HTRU2

Courtesy

Dr Robert Lyon

University of Manchester

School of Physics and Astronomy

In [ ]:
import pandas as pd
import numpy as np
train_df= pd.read_csv("https://student-datasets-bucket.s3.ap-south-1.amazonaws.com/whitehat-ds-datasets/project-5/pulsar-star-prediction-train.csv")
test_df= pd.read_csv("https://student-datasets-bucket.s3.ap-south-1.amazonaws.com/whitehat-ds-datasets/project-5/pulsar-star-prediction-test.csv")
train_df.head()

,target_class,Mean of the integrated profile,Standard deviation of the integrated profile,Excess kurtosis of the integrated profile,Skewness of the integrated profile,Mean of the DM-SNR curve,Standard deviation of the DM-SNR curve,Excess kurtosis of the DM-SNR curve,Skewness of the DM-SNR curve
0,0,111.109375,53.131064,0.280253,-0.222447,3.011706,20.355820,7.585482,62.383270
1,0,151.945312,47.973350,-0.250834,0.275367,2.115385,14.195484,11.640297,173.592172
2,1,52.335938,34.775008,2.478375,10.179171,8.230769,34.775947,4.652342,21.987882
3,0,121.562500,48.569498,-0.033391,-0.323514,2.595318,15.089924,8.734079,98.584122
4,0,133.664062,59.137852,-0.164198,-0.552877,1.542642,12.052034,12.226623,196.522501


In [ ]:
print("Thue number of rows and columns in train dataset are", train_df.shape)
print("The number of rows and columns in the test datset are", test_df.shape)

Thue number of rows and columns in train dataset are (11991, 9)
The number of rows and columns in the test datset are (5907, 9)


In [ ]:
#No missing values
train_df.isnull().sum()

target_class                                     0
 Mean of the integrated profile                  0
 Standard deviation of the integrated profile    0
 Excess kurtosis of the integrated profile       0
 Skewness of the integrated profile              0
 Mean of the DM-SNR curve                        0
 Standard deviation of the DM-SNR curve          0
 Excess kurtosis of the DM-SNR curve             0
 Skewness of the DM-SNR curve                    0
dtype: int64

In [ ]:
test_df.isnull().sum()

target_class                                     0
 Mean of the integrated profile                  0
 Standard deviation of the integrated profile    0
 Excess kurtosis of the integrated profile       0
 Skewness of the integrated profile              0
 Mean of the DM-SNR curve                        0
 Standard deviation of the DM-SNR curve          0
 Excess kurtosis of the DM-SNR curve             0
 Skewness of the DM-SNR curve                    0
dtype: int64

In [ ]:
print(train_df['target_class'].value_counts())
print(test_df['target_class'].value_counts())

0    10878
1     1113
Name: target_class, dtype: int64
0    5381
1     526
Name: target_class, dtype: int64


In [ ]:
#Feature Variables Extraction
x_train= train_df.iloc[:, 1:]
x_train.head()

,Mean of the integrated profile,Standard deviation of the integrated profile,Excess kurtosis of the integrated profile,Skewness of the integrated profile,Mean of the DM-SNR curve,Standard deviation of the DM-SNR curve,Excess kurtosis of the DM-SNR curve,Skewness of the DM-SNR curve
0,111.109375,53.131064,0.280253,-0.222447,3.011706,20.355820,7.585482,62.383270
1,151.945312,47.973350,-0.250834,0.275367,2.115385,14.195484,11.640297,173.592172
2,52.335938,34.775008,2.478375,10.179171,8.230769,34.775947,4.652342,21.987882
3,121.562500,48.569498,-0.033391,-0.323514,2.595318,15.089924,8.734079,98.584122
4,133.664062,59.137852,-0.164198,-0.552877,1.542642,12.052034,12.226623,196.522501


In [ ]:
x_test= test_df.iloc[:, 1:]
x_test.head()

,Mean of the integrated profile,Standard deviation of the integrated profile,Excess kurtosis of the integrated profile,Skewness of the integrated profile,Mean of the DM-SNR curve,Standard deviation of the DM-SNR curve,Excess kurtosis of the DM-SNR curve,Skewness of the DM-SNR curve
0,116.906250,48.920605,0.186046,-0.129815,3.037625,17.737102,8.122621,78.813405
1,75.585938,34.386254,2.025498,8.652913,3.765050,21.897049,7.048189,55.878791
2,103.273438,46.996628,0.504295,0.821088,2.244983,15.622566,9.330498,105.134941
3,101.078125,48.587487,1.011427,1.151870,81.887960,81.464136,0.485105,-1.117904
4,113.226562,48.608804,0.291538,0.292120,6.291806,26.585056,4.540138,21.708268


In [ ]:
y_train= train_df.iloc[:, :1]
y_train.head()

,target_class
0,0
1,0
2,1
3,0
4,0


In [ ]:
y_test= test_df.iloc[:, :1]
y_test.head()

,target_class
0,0
1,1
2,0
3,1
4,0


In [ ]:
import xgboost as xg
from sklearn.metrics import confusion_matrix,classification_report
model=xg.XGBClassifier() 
model.fit(x_train, y_train)
pred=model.predict(x_test)
print(pred)

/usr/local/lib/python3.7/dist-packages/sklearn/preprocessing/_label.py:98: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.7/dist-packages/sklearn/preprocessing/_label.py:133: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[0 1 0 ... 0 0 0]


In [ ]:
confusion_matrix(y_test,pred)

array([[5348,   33],
       [  82,  444]])

In [ ]:
precision= 444/ (444+ 33)
print("the precision is", precision)
recall = 444/(444+82)
print("The recalll is", recall)
f1_score= 2*((precision*recall)/ (precision+recall))
print("the f1 score is", f1_score)
print(classification_report(y_test, pred))

the precision is 0.9308176100628931
The recalll is 0.844106463878327
the f1 score is 0.8853439680957128
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      5381
           1       0.93      0.84      0.89       526

    accuracy                           0.98      5907
   macro avg       0.96      0.92      0.94      5907
weighted avg       0.98      0.98      0.98      5907

